# Week 06 Lab 02: Create and use MCP server.  
Marketplaces for MCP servers.

https://glama.ai/mcp  
https://smithery.ai/servers  
Creating an MCP server is pretty simple, but not super-simple. The excitement around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.  
Let's review some python code made mostly by a hard-working Engineering Team:

backend/accounts.py

In [2]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override = True)

True

In [3]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.

import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog = subprocess.DEVNULL)

In [6]:
from backend.accounts import Account

In [7]:
account = Account.get('fa')
account.reset()
account

Account(name='fa', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [8]:
account.buy_shares("AMZN", 3, "Because this book store website looks promising.")

'Completed. Latest details:\n{"name": "fa", "balance": 9214.3819, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 261.8727, "timestamp": "2026-07-13 10:18:18", "rationale": "Because this book store website looks promising."}], "portfolio_value_time_series": [["2026-07-13 10:18:18", 9998.4319]], "total_portfolio_value": 9998.4319, "total_profit_loss": -1.5681000000004133}'

In [9]:
account.report()

'{"name": "fa", "balance": 9214.3819, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 261.8727, "timestamp": "2026-07-13 10:18:18", "rationale": "Because this book store website looks promising."}], "portfolio_value_time_series": [["2026-07-13 10:18:18", 9998.4319], ["2026-07-13 10:18:25", 9998.4619]], "total_portfolio_value": 9998.4619, "total_profit_loss": -1.5380999999997584}'

In [10]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 261.8727,
  'timestamp': '2026-07-13 10:18:18',
  'rationale': 'Because this book store website looks promising.'}]

## Write an MCP server and use it directly.  

In [11]:
# Now let's use our accounts server as an MCP server
params = {'command' : 'uv', 'args' : ['run', '-m', 'backend.accounts_server']}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

In [12]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=

In [13]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is fa and my account is under the name fa. What's my balance and my holdings?"
model = 'gpt-4o-mini'

In [14]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name= 'account_manager', instructions= instructions, model = model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Your account balance is **$9,214.38**. 

In terms of holdings, you have **3 shares of Amazon (AMZN)**.